# ro-id — generare dataset + antrenare PaddleOCR (Colab)

1. **Generează** imagini sintetice CI (`generate.py`)
2. **Antrenează** modelul (`train_and_deploy.py`)
3. Descarcă **`model_export.zip`**

**Runtime → GPU** (pentru antrenare; generarea merge și pe CPU, dar GPU e recomandat).

Salvează datasetul pe **Google Drive** — spațiul din `/content` se șterge la deconectare.

In [ ]:
# === Configurare (EDITEAZĂ AICI) ===
REPO_URL = "https://github.com/YOUR_ORG/ro-id.git"
REPO_BRANCH = "main"
WORKDIR = "/content/ro-id"

# --- Dataset ---
GENERATE_DATASET = True           # True = rulează generate.py în Colab
SKIP_GENERATION_IF_EXISTS = True  # sare generarea dacă există labels/train.txt
GENERATE_COUNT = 5000             # 5k–20k rezonabil pe Colab; 50k+ pe Drive cu răbdare
GENERATE_BATCH_SIZE = 200
GENERATE_WORKERS = 2              # ~2 CPU în Colab

DATASET_DIR = "/content/drive/MyDrive/ro-id/dataset_colab"  # recomandat: Drive
MOUNT_GOOGLE_DRIVE = True
UPLOAD_DATASET_ZIP = False        # alternativ: încarcă ZIP cu dataset deja generat

# --- Antrenare ---
EPOCHS = 15
BATCH_SIZE = 128
USE_FAST = False
SKIP_CHECK_DATASET = False
INSTALL_PADDLEOCR_REPO = True
MODEL_NAME = "frc_ci_rec"

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

def run(cmd, *, cwd=None, env=None, check=True):
    print(">", " ".join(str(c) for c in cmd))
    return subprocess.run(cmd, cwd=cwd, env=env, check=check)

try:
    import torch
    print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'nu (OK pentru generare)'}")
except Exception:
    !nvidia-smi 2>/dev/null || true

In [ ]:
if Path(WORKDIR).is_dir():
    print(f"Repo există: {WORKDIR}")
else:
    run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, WORKDIR])

REPO_ROOT = Path(WORKDIR).resolve()
COLAB_ROOT = REPO_ROOT / "colab"
assert (REPO_ROOT / "generate.py").is_file(), "Lipsește generate.py"
assert (REPO_ROOT / "scripts" / "train_and_deploy.py").is_file()
print("Repo:", REPO_ROOT)

In [ ]:
if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
# Mediu Python (același .venv pentru generare + antrenare)
VENV_PY = REPO_ROOT / ".venv" / "bin" / "python"
if not VENV_PY.is_file():
    run([sys.executable, "-m", "venv", str(REPO_ROOT / ".venv")])

run([str(VENV_PY), "-m", "pip", "install", "-U", "pip", "wheel", "setuptools"])
run([str(VENV_PY), "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")])
print("Dependențe generator instalate.")

In [ ]:
dataset_path = Path(DATASET_DIR)
train_labels = dataset_path / "labels" / "train.txt"

if UPLOAD_DATASET_ZIP:
    from google.colab import files
    zip_name = next(iter(files.upload().keys()))
    extract_to = REPO_ROOT / "dataset_upload"
    if extract_to.exists():
        shutil.rmtree(extract_to)
    extract_to.mkdir(parents=True)
    shutil.unpack_archive(zip_name, extract_to)
    if (extract_to / "dataset" / "labels" / "train.txt").is_file():
        dataset_path = extract_to / "dataset"
    elif (extract_to / "labels" / "train.txt").is_file():
        dataset_path = extract_to
    else:
        raise SystemExit("ZIP fără labels/train.txt")
    train_labels = dataset_path / "labels" / "train.txt"

elif GENERATE_DATASET:
    if SKIP_GENERATION_IF_EXISTS and train_labels.is_file():
        n = sum(1 for ln in train_labels.read_text(encoding="utf-8").splitlines() if ln.strip())
        print(f"Dataset existent pe Drive ({n} etichete) — sar generarea.")
    else:
        dataset_path.mkdir(parents=True, exist_ok=True)
        print(f"Generare {GENERATE_COUNT} imagini → {dataset_path}")
        run([
            str(VENV_PY), str(REPO_ROOT / "generate.py"),
            "--count", str(GENERATE_COUNT),
            "--output", str(dataset_path),
            "--batch-size", str(GENERATE_BATCH_SIZE),
            "--workers", str(GENERATE_WORKERS),
        ], cwd=REPO_ROOT)
        train_labels = dataset_path / "labels" / "train.txt"

if not train_labels.is_file():
    raise SystemExit(
        f"Lipsește {train_labels}. Setează GENERATE_DATASET=True sau UPLOAD_DATASET_ZIP=True."
    )
if not (dataset_path / "images").is_dir():
    raise SystemExit(f"Lipsește {dataset_path / 'images'}")

n_labels = sum(1 for ln in train_labels.read_text(encoding="utf-8").splitlines() if ln.strip())
print(f"Dataset gata: {dataset_path} ({n_labels} sample-uri)")

In [ ]:
# Paddle + dependențe antrenare
run([str(VENV_PY), "-m", "pip", "install", "paddlepaddle-gpu>=3.2.0,<3.3.0",
     "-i", "https://www.paddlepaddle.org.cn/packages/stable/cu123/"])
run([str(VENV_PY), "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements-train.txt")])
run([str(VENV_PY), "-m", "pip", "install", "paddleocr>=3.0.0,<4.0.0"])
run([str(VENV_PY), "-c", "import paddle; print('Paddle', paddle.__version__, 'CUDA', paddle.is_compiled_with_cuda())"])

In [ ]:
dataset_rel = os.path.relpath(dataset_path.resolve(), REPO_ROOT)
cmd = [str(VENV_PY), str(REPO_ROOT / "scripts" / "train_and_deploy.py"),
       "--dataset", dataset_rel, "--device", "gpu:0",
       "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE),
       "--paddle-python", str(VENV_PY), "--model-name", MODEL_NAME,
       "--skip-deploy"]
if USE_FAST: cmd.append("--fast")
if SKIP_CHECK_DATASET: cmd.append("--skip-check-dataset")
if INSTALL_PADDLEOCR_REPO: cmd.append("--install-paddleocr-repo")

env = os.environ.copy()
env.update({"PADDLE_DEVICE": "gpu:0", "FLAGS_use_mkldnn": "0", "MPLBACKEND": "Agg",
            "PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK": "True"})
run(cmd, cwd=REPO_ROOT, env=env)

In [ ]:
sys.path.insert(0, str(COLAB_ROOT))
from package_model_export import package_model_export

MODEL_DIR = REPO_ROOT / "exports" / MODEL_NAME
ZIP_PATH = COLAB_ROOT / "model_export.zip"
package_model_export(model_dir=MODEL_DIR, zip_path=ZIP_PATH)
print(f"{ZIP_PATH} ({ZIP_PATH.stat().st_size / 1_048_576:.1f} MB)")

In [ ]:
from google.colab import files
files.download(str(ZIP_PATH))
print("Copiază model_export.zip în FRCHub/services/paddle-ocr/models/")

## Note

- **Prima rulare:** generează + antrenează (ore, în funcție de `GENERATE_COUNT` și `EPOCHS`).
- **Rulări următoare:** cu `SKIP_GENERATION_IF_EXISTS=True`, refolosește datasetul de pe Drive.
- Template-urile specimen sunt în repo (`templates/*.png`) — nu ai nevoie de PC pentru generare.
- Pentru 50k imagini: crește `GENERATE_COUNT`, asigură spațiu pe Drive (~10–30 GB).